In [ ]:
import geopandas as gpd
import pandas as pd
import folium
from shapely.ops import unary_union
from shapely import MultiLineString

# creating the bikepath polygon - represents 300m of every bikepath on the city

# get all bikepaths
GDF_saopaulo = gpd.read_file('./data/sao_paulo_demographics.geojson')
bikepaths_GDF = gpd.read_file('./data/Ciclorrotas.shp')
bikelanes_GDF = gpd.read_file('./data/Ciclovias.shp')
today_bikelanes_GDF = pd.concat([bikepaths_GDF, bikelanes_GDF])

# clear gdfs from multilinestrings, they stop later processes
multilinestring_rows = today_bikelanes_GDF.geometry.apply(lambda geom: isinstance(geom, MultiLineString))
today_bikelanes_GDF = today_bikelanes_GDF[~multilinestring_rows]
today_bikelanes_GDF = today_bikelanes_GDF.reset_index()

# switch crs modes, create 300m buffer around all bikepaths
today_bikelanes_GDF = today_bikelanes_GDF.set_crs(epsg=4674)
today_bikelanes_GDF = today_bikelanes_GDF.to_crs(epsg=31983) # web mercator, 1:1 com metros
today_bikelanes_GDF_bufferzone = gpd.GeoDataFrame(geometry=[])
today_bikelanes_GDF_bufferzone['geometry'] = today_bikelanes_GDF.geometry.buffer(300)
today_bikelanes_GDF_bufferzone = today_bikelanes_GDF_bufferzone.set_crs(epsg=31983)

# make buffer polygons into one single entity
today_bikelanes_merged_polygon = unary_union(today_bikelanes_GDF_bufferzone.geometry)

# make the area polygon into a GDF and assign a crs for it 
GDF_today_and_theorized_polygon_merge = gpd.GeoDataFrame(geometry=[today_bikelanes_merged_polygon])

# finding census zones intersecting the bikelane polygon
GDF_intersection_zones_gdfmerge = gpd.overlay(GDF_saopaulo, GDF_today_and_theorized_polygon_merge, how="intersection")



# removing non-intersecting zones
GDF_saopaulo_overlaid_zones = GDF_saopaulo[GDF_saopaulo['id'].isin(GDF_intersection_zones_gdfmerge['id'])]

# matching both gdfs order
GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.sort_values('id', ascending = True)
GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.sort_values('id', ascending = True)
GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.reset_index()
GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.reset_index()

# calculating the estimate population in said intersected zones (accounting for parcial intersection)
GDF_intersection_zones_gdfmerge['areas_ratio'] = GDF_intersection_zones_gdfmerge.to_crs('EPSG:31983').area/GDF_saopaulo_overlaid_zones.to_crs('EPSG:31983').area
GDF_intersection_zones_gdfmerge['estimate_population'] = GDF_intersection_zones_gdfmerge['populacao'] * GDF_intersection_zones_gdfmerge['areas_ratio']

GDF_intersection_zones_gdfmerge['estimate_population'].sum()

C:\Users\João Rahal\AppData\Local\Temp\ipykernel_15240\2567769647.py:34: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:31983
Right CRS: None

  GDF_intersection_zones_gdfmerge = gpd.overlay(GDF_saopaulo, GDF_today_and_theorized_polygon_merge, how="intersection")


KeyboardInterrupt: 

In [ ]:
3642451.106460753

5385878.792945537

In [ ]:
import json
from shapely.geometry import LineString

file_name = ".//data//caminhos_bike_3000m_11.geojson"
with open(file_name, 'r', encoding='utf-8') as file:
    data = json.load(file)

fmap = folium.Map(location=[-23.5, -46.6], zoom_start=10)

i = 0
new_lanes_GDF = gpd.GeoDataFrame(geometry=[])

while i < len(data['features']):
    coordinates = data['features'][i]['properties']['paths'][0]['points']['coordinates']
    linestring_coordinates = LineString([(point[0], point[1]) for point in coordinates])
    temp_gdf = gpd.GeoDataFrame(geometry=[linestring_coordinates])
    new_lanes_GDF = pd.concat([new_lanes_GDF , temp_gdf], ignore_index=True)
    i += 1

new_lanes_GDF = new_lanes_GDF.set_crs(epsg=4326)
new_lanes_GDF = new_lanes_GDF.to_crs(epsg=31983)

today_bikelanes_multilinestring = unary_union(today_bikelanes_GDF.geometry)
new_lanes_GDF = new_lanes_GDF.geometry.apply(lambda line: line.difference(today_bikelanes_multilinestring))

new_lanes_GDF_bufferzone = gpd.GeoDataFrame(geometry=[])
new_lanes_GDF_bufferzone['geometry'] = new_lanes_GDF.geometry.buffer(300)
new_lanes_GDF_bufferzone = new_lanes_GDF_bufferzone.set_crs(epsg=31983)
new_lanes_GDF_bufferzone = new_lanes_GDF_bufferzone.to_crs(epsg=4326)

# make buffer polygons into one single entity
theorized_bikelanes_merged_polygon = unary_union(new_lanes_GDF_bufferzone.geometry)

# make the area polygon into a GDF and assign a crs for it 
GDF_today_and_theorized_polygon_merge = gpd.GeoDataFrame(geometry=[theorized_bikelanes_merged_polygon])

# finding census zones intersecting the bikelane polygon
GDF_intersection_zones_gdfmerge = gpd.overlay(GDF_saopaulo, GDF_today_and_theorized_polygon_merge, how="intersection")

# removing non-intersecting zones
GDF_saopaulo_overlaid_zones = GDF_saopaulo[GDF_saopaulo['id'].isin(GDF_intersection_zones_gdfmerge['id'])]

# matching both gdfs order
GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.sort_values('id', ascending = True)
GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.sort_values('id', ascending = True)
GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.reset_index()
GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.reset_index()

# calculating the estimate population in said intersected zones (accounting for parcial intersection)
GDF_intersection_zones_gdfmerge['areas_ratio'] = GDF_intersection_zones_gdfmerge.to_crs('EPSG:31983').area/GDF_saopaulo_overlaid_zones.to_crs('EPSG:31983').area
GDF_intersection_zones_gdfmerge['estimate_population'] = GDF_intersection_zones_gdfmerge['populacao'] * GDF_intersection_zones_gdfmerge['areas_ratio']

GDF_intersection_zones_gdfmerge['estimate_population'].sum()


C:\Users\João Rahal\AppData\Local\Temp\ipykernel_2160\149078274.py:38: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:31983
Right CRS: None

  GDF_intersection_zones_gdfmerge = gpd.overlay(GDF_saopaulo, GDF_today_and_theorized_polygon_merge, how="intersection")


0.0